# Install Of Dependencies
---
This libraries do not usually come pre-installed.

In [ ]:
!pip install umap-learn
!pip install pytorch-msssim

# Imports
---
We use the following libraries:
- Torch: Model creation, loss functions, optimizers...
- Matplotlib: graphics and plots.
- Numpy: handling tensors and mathematical operations.
- Umap: UMAP representation of the latent space.
- Pytorch-mssim: SSIM implementation compatible with pytorch.

In [70]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torchvision.transforms import InterpolationMode


import matplotlib.pyplot as plt
import numpy as np

import umap
from pytorch_msssim import ssim

# Autoencoder Class
---
- For this experiment, the most interesting part is the encoder. It extracts the information from the image and transforms it into the latent vector.
- The decoder can be designed in different ways, but in this case, I've opted for the most balanced approach, making it the inverse of the encoder. 
- The autoencoder is a combination of an encoder and a decoder.

In [71]:
class MNISTEncoder(nn.Module):
    def __init__(self, act_function = nn.ReLU, latent_act_function = None, dropout_rate = 0.2, latent_dim = 64):
        super(MNISTEncoder, self).__init__()

        self.convnet = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(4),
            act_function(),
            nn.Conv2d(4, 8, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(8),
            act_function(),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(16),
            act_function(),
        )

        self.linear = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 7 * 7, latent_dim * 4),
            nn.BatchNorm1d(latent_dim * 4),
            act_function(),
            nn.Dropout(dropout_rate),
            nn.Linear(latent_dim * 4, latent_dim)
        )

        if latent_act_function: self.linear.append(latent_act_function())

    def forward(self, x):
        x = self.convnet(x)
        x = self.linear(x)
        return x

class MNISTDecoder(nn.Module):
    def __init__(self, act_function=nn.ReLU, dropout_rate=0.2, latent_dim=64):
        super(MNISTDecoder, self).__init__()

        self.linear = nn.Sequential(
            nn.Linear(latent_dim, latent_dim * 4),
            nn.BatchNorm1d(latent_dim * 4),
            act_function(),
            nn.Dropout(dropout_rate),
            nn.Linear(latent_dim * 4, 16 * 7 * 7),
            nn.Unflatten(dim=1, unflattened_size=(16, 7, 7))
        )

        self.deconvnet = nn.Sequential(
            nn.ConvTranspose2d(16, 8, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(8),
            act_function(),
            nn.ConvTranspose2d(8, 4, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(4),
            act_function(),
            nn.ConvTranspose2d(4, 1, kernel_size=3, stride=1, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.linear(x)
        x = self.deconvnet(x)
        return x

class MNISTAutoencoder(nn.Module):
    def __init__(self, act_function=nn.ReLU, latent_act_function=None, dropout_rate=0.2, latent_dim=64):
        super(MNISTAutoencoder, self).__init__()

        self.encoder = MNISTEncoder(
            act_function=act_function,
            latent_act_function=latent_act_function,
            dropout_rate=dropout_rate,
            latent_dim=latent_dim
        )
        self.decoder = MNISTDecoder(
            act_function=act_function,
            dropout_rate=dropout_rate,
            latent_dim=latent_dim
        )

    def forward(self, x):
        z = self.encoder(x)
        reconstructed = self.decoder(z)
        return reconstructed

# Parameters
---

In [72]:
BATCH_SIZE = 1024

RUNS = 1

# Data Loaders
---
### Transform
RandomAffine is used to apply random transformations to the images. This helps prevent overfitting, improves generalization, and increases the difficulty of the original MNIST dataset by introducing additional variability.

### Splits
Four different splits are made:

- Train: 80% of the dataset.
- Val: 10% of the dataset.
- Test: 10% of the dataset.
- Full: the whole dataset is used. This split is used to give image examples and other evaluation metrics.

In [ ]:
NUM_WORKERS = os.cpu_count() // 2

transform = transforms.Compose([
    transforms.RandomAffine(
        degrees=10,
        translate=(0.04, 0.04),
        scale=(0.96, 1.04),
        interpolation=InterpolationMode.BILINEAR
    ),
    transforms.ToTensor()
])

full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)

total_size = len(full_train_dataset)
train_size = int(total_size * 0.8)
val_size = int(total_size * 0.1)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_train_dataset, [train_size, val_size, test_size])

full_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2)

val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Auxiliary Functions
---

All the functions have been adapted to work with the configs as a way to automate the work.

### Train Function
---

In [74]:
def model_train(model,
          model_id,
          criterion,
          lambda_l1,
          epochs,
          log):

    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    train_losses = []
    val_losses = []

    print(f"Started training: {model_id}")
    for epoch in range(epochs):
        model.train()
        running_train_loss = 0.0

        for images, _ in train_loader:
            images = images.to(device)

            latent_vectors = model.encoder(images)
            reconstructed = model.decoder(latent_vectors)
            recon_loss = criterion(reconstructed, images)

            l1_penalty = torch.mean(torch.abs(latent_vectors))

            loss = recon_loss + lambda_l1 * l1_penalty

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_train_loss += loss.item()

        train_losses.append(running_train_loss / len(train_loader))


        model.eval()
        running_val_loss = 0.0

        with torch.no_grad():
            for images, _ in val_loader:
                images = images.to(device)

                latent_vectors = model.encoder(images)
                reconstructed = model.decoder(latent_vectors)
                recon_loss = criterion(reconstructed, images)

                l1_penalty = torch.mean(torch.abs(latent_vectors))

                loss = recon_loss + lambda_l1 * l1_penalty

                running_val_loss += loss.item()

        val_losses.append(running_val_loss / len(val_loader))

        if log: print(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_losses[epoch]:.4f} | Val Loss: {val_losses[epoch]:.4f}")

    return train_losses, val_losses


    #torch.save(model.state_dict(), "mnist_autoencoder.pth")

def config_train(configs, log):
    return [model_train(model = config["model"],
                model_id = config["id"],
                criterion = config["loss_function"],
                lambda_l1 = config["lambda_l1"],
                epochs = config["epochs"],
                log = log) for config in configs]


### Train Losses Graph Function
---

In [75]:
def losses_analysis(configs):
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    for i, config in enumerate(configs):
        axes[0].plot(config["train_loss"][3:], label=config["id"])
        axes[1].plot(config["val_loss"][3:], label=config["id"])

    axes[0].set_title("Train Loss (Average)")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.7)

    axes[1].set_title("Validation Loss (Average)")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

### Test Losses Comparison Graph Function
---

In [76]:
def model_test_eval(config, criterions):
    config["model"].eval()

    losses = {}
    batch_size = 0
    
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            reconstructed = config["model"](images)

            for name, f in criterions.items():
                if name not in losses:
                    losses[name] = f(reconstructed, images).item() * images.size(0)
                else:
                    losses[name] += f(reconstructed, images).item() * images.size(0)

            batch_size += images.size(0)

    for k in losses:
        losses[k] /= batch_size

    return losses.copy()
    

def config_test_eval(configs, criterions):
    return [model_test_eval(config, criterions) for config in configs]    

### Neuron Analytics Function
---
Visualization of the average and variance per neuron.

In [91]:
def neuron_analysis(model, model_id, latent_dim):
    sum_latent = torch.zeros(latent_dim).to(device)
    sum_sq_latent = torch.zeros(latent_dim).to(device)
    total_samples = 0

    with torch.no_grad():
        for images, _ in full_loader:
            images = images.to(device)

            latent_vectors = model.encoder(images)
            sum_latent += torch.sum(latent_vectors, dim=0)
            sum_sq_latent += torch.sum(latent_vectors ** 2, dim=0)
            total_samples += images.size(0)

    mean_latent = sum_latent / total_samples
    variance_latent = (sum_sq_latent / total_samples) - (mean_latent ** 2)

    mean_list = mean_latent.cpu().tolist()
    variance_list = variance_latent.cpu().tolist()

    fig, axes = plt.subplots(1, 2, figsize = (10, 5))
    plt.suptitle(f"Model: {model_id}")

    axes[0].set_title("Average per neuron in the latent space")
    axes[0].plot(range(latent_dim), mean_list, color='blue', marker='o', markersize=3)
    axes[0].set_xlabel("Index")
    axes[0].set_ylabel("Value")
    axes[0].grid(True, alpha=0.3)

    axes[1].set_title("Variance per neuron in latent space")
    axes[1].plot(range(latent_dim), variance_list, color='red', marker='o', markersize=3)
    axes[1].set_xlabel("Index")
    axes[1].set_ylabel("Value")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

def config_neuron_analysis(configs):
    for config in configs:
        neuron_analysis(config["model"],
                       config["id"],
                       config["latent_dim"])

### UMAP Analysis
---
Custom functions to visualize de latent vectors as 2D graphs.

In [78]:
def umap_analysis(model, model_id):
    latent_vectors_list = []
    labels_list = []

    with torch.no_grad():
        for images, labels in test_loader:
            latent_vector = model.encoder(images.to(device))

            latent_vectors_list.append(latent_vector.cpu().numpy())
            labels_list.append(labels.numpy())

    latent_vectors = np.concatenate(latent_vectors_list, axis=0)
    y_test = np.concatenate(labels_list, axis=0)

    reducer = umap.UMAP(n_neighbors=30, min_dist=0.05, random_state=42)
    codes_2d = reducer.fit_transform(latent_vectors)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        codes_2d[:, 0],
        codes_2d[:, 1],
        c=y_test,
        cmap="tab10",
        s=1,
        alpha=0.7,
    )
    plt.colorbar(scatter, label="Digit")
    plt.title(f"{model_id} latent vector")
    plt.grid(True, alpha=0.2)
    plt.show()


def config_umap_analysis(configs):
    for config in configs:
        umap_analysis(config["model"], config["id"])

### Custom Loss Functions
---
- ssim_loss: SSIM adaptation
- SSIML1Loss: combination of SSIM and L1 losses using a factor alpha.

In [79]:
def ssim_loss(reconstructed, images):
    return 1 - ssim(reconstructed, images, data_range = 1, size_average = True, win_size=5)

In [80]:
class SSIML1Loss:
    def __init__(self, alpha):
        self.alpha = alpha

        self.l1 = nn.L1Loss()

    def __call__(self, reconstructed, images):
        return ssim_loss(reconstructed, images) * (1 - self.alpha) + self.l1(reconstructed, images) * self.alpha

# Configuration Of The Models
---
Each line of the list `configs` is an individual model. The configuration follows the following pattern:

```python
{
    "id": str,
    "act_function": function (activation function to use between the convolutions),
    "latent_act_function": function (activation function to use in the output of the encoder),
    "dropout_rate": float,
    "latent_dim": int (dimension of the latent space),
    "loss_function": class/function,
    "lambda_l1": float (strenght of the L1 regularization),
    "epochs": int
}
```

In [81]:
configs = [
    {"id": "32", "act_function": nn.SiLU, "latent_act_function": nn.Tanh, "dropout_rate": 0.0, "latent_dim": 32, "loss_function": SSIML1Loss(0.4), "lambda_l1": 1e-10, "epochs": 19},
]


# Training
---

### Criterion Configuration
---
Here, the criterions used to test and compare the models are chosen.

In [82]:
CRITERIONS = {
    "MSE": nn.MSELoss(),
    "L1": nn.L1Loss(),
    "SSIM": ssim_loss
}

### Running!
---
Here all the auxiliary functions are used to train each model in the `configs` list and save the statistics. The models are trained sequentially. The following data is saved for each run:

- Train loss
- Val loss
- Test loss: for each criterion given.

This code is executed once per run until the limit `RUNS` is reached.

In [ ]:
ids = [config["id"] for config in configs]
results = {}

for id in ids:
    results[id] = {
        "train_loss": [],
        "val_loss": [],
        "test_loss": {crit: [] for crit in CRITERIONS}
    }

for i in range(RUNS):
    
    print(f"[{i}] Creating models...")
    
    for j in range(len(configs)):
        configs[j]["model"] = MNISTAutoencoder(
            act_function=configs[j]["act_function"],
            latent_act_function=configs[j]["latent_act_function"],
            dropout_rate=configs[j]["dropout_rate"],
            latent_dim=configs[j]["latent_dim"]
        ).to(device)

    print(f"[{i}] Training models...")
    train_results = config_train(configs, False)

    print(f"[{i}] Evaluating models...")
    test_results = config_test_eval(configs, CRITERIONS)

    for id, r in zip(ids, train_results):
        results[id]["train_loss"].append(r[0])
        results[id]["val_loss"].append(r[1])

    for id, r in zip(ids, test_results):
        for crit, value in r.items():
            results[id]["test_loss"][crit].append(value)



Here the losses are averaged across all runs to ensure the reliability of the results.

In [84]:
for id, config in zip(ids, configs):
    config["train_loss"] = np.mean(results[id]["train_loss"], axis=0)
    config["val_loss"] = np.mean(results[id]["val_loss"], axis=0)
    config["test_loss"] = {c: np.mean(v) for c, v in results[id]["test_loss"].items()}

# Results And Analytics
---

## Training Loss Graph
---
Loss graph for each model and both training and validation splits. The first three epochs have been  cutted to ensure readability.

In [ ]:
losses_analysis(configs)

## Test Loss Comparison Graph
---
Loss graph for each model in the given criterions on the test split. This allows fair comparison.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, criterion in zip(axes, CRITERIONS):
    ax.set_title(criterion)

    losses = [config["test_loss"][criterion] for config in configs]
    ax.plot(ids, losses, label = criterion)

plt.show()

## UMAP Visualization
---

In [ ]:
config_umap_analysis(configs)

## Neuron Analytics Visualization
---
Visualization of the average and variance per neuron. Is more useful in sparse autoencoders.

In [ ]:
config_neuron_analysis(configs)